In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import classification_report, cohen_kappa_score, accuracy_score, mean_absolute_error
from sklearn.utils.class_weight import compute_class_weight
import os
import zipfile

PROCESSED_DIR = '../data/processed/'
df_train = pd.read_parquet(PROCESSED_DIR + 'train_cleaned.parquet')
df_valid = pd.read_parquet(PROCESSED_DIR + 'valid_cleaned.parquet')

df_train['label'] = df_train['label'].astype(np.int64)
df_valid['label'] = df_valid['label'].astype(np.int64)

datasets = DatasetDict({
    'train': Dataset.from_pandas(df_train[['Sentence_Normalized', 'label']]),
    'valid': Dataset.from_pandas(df_valid[['Sentence_Normalized', 'label']]),
})
print(f" Dữ liệu sẵn sàng! Train: {len(df_train)} | Valid: {len(df_valid)}")

✅ Dữ liệu sẵn sàng! Train: 54626 | Valid: 7310 | Test: 7286


In [ ]:
MODEL_NAME = "aubmindlab/bert-base-arabertv02"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["Sentence_Normalized"], 
        padding="max_length", 
        truncation=True, 
        max_length=128  
    )

print("⏳ Đang Tokenize bằng AraBERTv2...")
tokenized_datasets = datasets.map(tokenize_function, batched=True)

# Format PyTorch Tensors
tokenized_datasets["train"].set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
tokenized_datasets["valid"].set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
print(" Hoàn tất Tokenize!")

⏳ Đang Tokenize bằng AraBERTv2...


Map:   0%|          | 0/54626 [00:00<?, ? examples/s]

Map:   0%|          | 0/7310 [00:00<?, ? examples/s]

Map:   0%|          | 0/7286 [00:00<?, ? examples/s]

✅ Hoàn tất Tokenize!


In [ ]:
print("⚖️ TÍNH TOÁN TRỌNG SỐ MẪU (SAMPLE-LEVEL WEIGHTS)...")
y_train = df_train['label'].values
cw = compute_class_weight('balanced', classes=np.arange(19), y=y_train)

cw_clipped = np.clip(cw, 0.5, 5.0)
cw_normalized = cw_clipped / cw_clipped.mean()
class_weights_tensor = torch.tensor(cw_normalized, dtype=torch.float32)

print("🔸 Class weights (đã clip và chuẩn hóa):")
print(np.round(cw_normalized, 2))

print("\n⚙️ KHỞI TẠO MÔ HÌNH VÀ LORA CHO CROSS-ENTROPY...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=19
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["query", "value", "dense"] 
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

⚖️ TÍNH TOÁN TRỌNG SỐ MẪU (SAMPLE-LEVEL WEIGHTS)...
🔸 Class weights (đã clip và chuẩn hóa):
[2.02 2.02 1.02 1.99 0.44 0.97 0.28 0.26 0.73 0.2  0.29 0.2  0.36 0.2
 0.58 1.35 2.02 2.02 2.02]

⚙️ KHỞI TẠO MÔ HÌNH VÀ LORA CHO CROSS-ENTROPY...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 2,398,483 || all params: 137,606,438 || trainable%: 1.7430


In [ ]:
class CrossEntropyTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits  
        device = logits.device
        
        # 1. Cross-Entropy Loss per sample
        loss_per_sample = F.cross_entropy(logits, labels, reduction='none')
        
        # 2. Apply class weights
        if self.class_weights is not None:
            w = self.class_weights.to(device)[labels]
            loss = (loss_per_sample * w).mean()
        else:
            loss = loss_per_sample.mean()
            
        return (loss, outputs) if return_outputs else loss

def compute_metrics_ce(eval_pred):
    logits, labels = eval_pred
    pred_labels = np.argmax(logits, axis=-1)
    
    qwk = cohen_kappa_score(labels, pred_labels, weights='quadratic')
    acc = accuracy_score(labels, pred_labels)
    
    return {"qwk": qwk, "accuracy": acc}

In [ ]:
training_args = TrainingArguments(
    output_dir="../saved_models/arabert_lora_ce",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,               
    per_device_train_batch_size=8,    
    gradient_accumulation_steps=4,    
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="qwk",
    greater_is_better=True,
    bf16=True,
    fp16=False,  
    seed=42
)

trainer = CrossEntropyTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["valid"],
    compute_metrics=compute_metrics_ce,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    class_weights=class_weights_tensor
)

print(" BẮT ĐẦU HUẤN LUYỆN WEIGHTED CROSS-ENTROPY...")
trainer.train()

trainer.save_model("../saved_models/arabert_lora_ce_best")
tokenizer.save_pretrained("../saved_models/arabert_lora_ce_best")

🚀 BẮT ĐẦU HUẤN LUYỆN WEIGHTED CROSS-ENTROPY...


  0%|          | 0/8535 [00:00<?, ?it/s]

{'loss': 0.8211, 'grad_norm': 14.279739379882812, 'learning_rate': 0.00028242530755711773, 'epoch': 0.29}
{'loss': 0.6707, 'grad_norm': 7.757061004638672, 'learning_rate': 0.00026485061511423544, 'epoch': 0.59}
{'loss': 0.6075, 'grad_norm': 9.810874938964844, 'learning_rate': 0.00024727592267135325, 'epoch': 0.88}


  0%|          | 0/457 [00:00<?, ?it/s]

{'eval_loss': 0.6002373695373535, 'eval_qwk': 0.7796124079270483, 'eval_accuracy': 0.47729138166894663, 'eval_runtime': 30.6558, 'eval_samples_per_second': 238.454, 'eval_steps_per_second': 14.907, 'epoch': 1.0}
{'loss': 0.5627, 'grad_norm': 20.126205444335938, 'learning_rate': 0.000229701230228471, 'epoch': 1.17}
{'loss': 0.532, 'grad_norm': 21.88242530822754, 'learning_rate': 0.00021212653778558875, 'epoch': 1.46}
{'loss': 0.5228, 'grad_norm': 5.144017696380615, 'learning_rate': 0.00019455184534270648, 'epoch': 1.76}


...

  0%|          | 0/457 [00:00<?, ?it/s]

{'eval_loss': 0.5659667253494263, 'eval_qwk': 0.8035217204357974, 'eval_accuracy': 0.5450068399452804, 'eval_runtime': 30.1631, 'eval_samples_per_second': 242.349, 'eval_steps_per_second': 15.151, 'epoch': 4.0}
{'loss': 0.3799, 'grad_norm': 5.388418674468994, 'learning_rate': 5.39543057996485e-05, 'epoch': 4.1}
{'loss': 0.3485, 'grad_norm': 5.0255818367004395, 'learning_rate': 3.6379613356766254e-05, 'epoch': 4.39}
{'loss': 0.3514, 'grad_norm': 11.585169792175293, 'learning_rate': 1.8804920913884008e-05, 'epoch': 4.69}
{'loss': 0.3407, 'grad_norm': 12.434958457946777, 'learning_rate': 1.2302284710017573e-06, 'epoch': 4.98}


  0%|          | 0/457 [00:00<?, ?it/s]

{'eval_loss': 0.5877799987792969, 'eval_qwk': 0.8011166262687301, 'eval_accuracy': 0.5422708618331054, 'eval_runtime': 29.9442, 'eval_samples_per_second': 244.121, 'eval_steps_per_second': 15.262, 'epoch': 5.0}
{'train_runtime': 3525.0324, 'train_samples_per_second': 77.483, 'train_steps_per_second': 2.421, 'train_loss': 0.48155621761596895, 'epoch': 5.0}


('../saved_models/arabert_lora_ce_best\\tokenizer_config.json',
 '../saved_models/arabert_lora_ce_best\\special_tokens_map.json',
 '../saved_models/arabert_lora_ce_best\\vocab.txt',
 '../saved_models/arabert_lora_ce_best\\added_tokens.json',
 '../saved_models/arabert_lora_ce_best\\tokenizer.json')

In [ ]:
print(" ĐANG DỰ ĐOÁN TRÊN TẬP VALIDATION...")
predictions_output = trainer.predict(tokenized_datasets["valid"])
logits = predictions_output.predictions  # Kích thước [7310, 19]
true_labels = predictions_output.label_ids.astype(int)

final_pred_labels = np.argmax(logits, axis=-1)

print("=== BÁO CÁO F1-SCORE CROSS-ENTROPY ===")
target_names = [f"Level_{i+1}" for i in range(19)]
print(classification_report(true_labels, final_pred_labels, target_names=target_names, zero_division=0))

print("=== CÁC CHỈ SỐ METRIC BAREC ===")
print(f"QWK (Main Metric)       : {cohen_kappa_score(true_labels, final_pred_labels, weights='quadratic'):.4f}")
print(f"Acc19 (Exact Match)     : {accuracy_score(true_labels, final_pred_labels):.4f}")
print(f"Adjacent Acc (±1 Level)  : {np.mean(np.abs(true_labels - final_pred_labels) <= 1):.4f}")
print(f"Avg Distance (MAE)      : {mean_absolute_error(true_labels, final_pred_labels):.4f}")

🔍 ĐANG DỰ ĐOÁN TRÊN TẬP VALIDATION...


  0%|          | 0/457 [00:00<?, ?it/s]


📊 === BÁO CÁO F1-SCORE CROSS-ENTROPY ===
              precision    recall  f1-score   support

     Level_1       0.73      0.80      0.76        44
     Level_2       0.60      0.81      0.69        68
     Level_3       0.50      0.74      0.60       182
     Level_4       0.34      0.64      0.45        78
     Level_5       0.56      0.59      0.58       417
     Level_6       0.47      0.57      0.52       189
     Level_7       0.61      0.66      0.63       701
     Level_8       0.69      0.64      0.66       613
     Level_9       0.44      0.69      0.54       236
    Level_10       0.72      0.78      0.75      1012
    Level_11       0.37      0.38      0.38       409
    Level_12       0.54      0.37      0.44      1491
    Level_13       0.40      0.48      0.44       349
    Level_14       0.58      0.53      0.55      1072
    Level_15       0.26      0.24      0.25       258
    Level_16       0.16      0.14      0.15       114
    Level_17       0.24      0.41      